# VISCERA — FINAL pipeline: **GastroNet-DINOv2 + VLM-Concept Teaching (Stage-1, GRL nuisance-suppression) + CG-AMIL @448 + SEMI + color-aug OOD**  (Run-All → ship)

Winning recipe (evidence: GastroNet-DINOv2 frozen-LP cross-center 0.93 ≫ generic DINOv3 0.835; exps/2 dinov2 hidden 0.854 > exps/3 dinov3 0.756).
**Method = 3 pillars:** (1) *Semi-supervised* Mean-Teacher+PU on the 144k pool; (2) *VLM-Concept Teaching* — Stage-1 distills 35 clinical concepts, diagnostic→trunk / nuisance→GRL (the OOD layer that ignores acquisition/scope shortcuts); (3) *OOD generalization* levers: color/FDA aug (`--aug domain`), MixStyle, WiSE-FT.
**Gate first** (cell 10, `RUN_GATE=True`): confirm the OOD bundle beats baseline on the held-out-center proxy before the single ship.

In [ ]:
import torch; print(torch.__version__)
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip -q install timm==1.0.27 scikit-learn

In [ ]:
REPO_URL = 'https://github.com/HuynhDoTanThanh/RARE2026.git'   # <-- your repo
%cd /content
!rm -rf rare && git clone $REPO_URL rare
%cd /content/rare

In [ ]:
# ---- HF token for the gated DINOv3 download (needed ONCE to fetch dinov3.pth; then cached to Drive) ----
# SECURE: put the token in Colab's Secrets panel (left sidebar, key icon) as name HF_TOKEN — do NOT paste it here
# (this notebook is pushed to GitHub; a committed token leaks publicly and GitHub push-protection will block it).
import os
try:
    from google.colab import userdata
    os.environ.setdefault('HF_TOKEN', userdata.get('HF_TOKEN'))   # read from Colab Secrets, never hard-coded
except Exception:
    pass   # not on Colab, or Secret not set -> fine if dinov3.pth is already cached on Drive
print('HF_TOKEN set' if os.environ.get('HF_TOKEN') else 'HF_TOKEN not set (ok if dinov3.pth already on Drive)')

In [ ]:
# ---- config + data (backbone weights + numbered data zips) ----
from google.colab import drive; drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/RARE_LG'   # <-- your Drive folder
BACKBONE  = 'dinov2'                            # 'dinov3' (ViT-B/16, stronger dense features + CG-AMIL) | 'dinov2' (exp1 path)
import glob, os, zipfile, shutil
# backbone weights: dinov2.pth (SSL teacher) or dinov3.pth (timm state_dict). Cache on Drive; dinov3 downloaded once.
if BACKBONE == 'dinov2':
    assert os.path.exists(f'{DRIVE_DIR}/dinov2.pth'), f'upload dinov2.pth to {DRIVE_DIR}'
    shutil.copy(f'{DRIVE_DIR}/dinov2.pth', 'dinov2.pth'); print('dinov2.pth OK')
else:
    if os.path.exists(f'{DRIVE_DIR}/dinov3.pth'):
        shutil.copy(f'{DRIVE_DIR}/dinov3.pth', 'dinov3.pth'); print('REUSED dinov3.pth from Drive')
    else:
        # gated download -> set os.environ['HF_TOKEN']='hf_...' in a cell ABOVE (do NOT commit the token to git)
        import timm, torch
        m = timm.create_model('vit_base_patch16_dinov3', pretrained=True, img_size=448, num_classes=0)
        torch.save(m.state_dict(), 'dinov3.pth'); shutil.copy('dinov3.pth', f'{DRIVE_DIR}/dinov3.pth')
        print('downloaded + cached dinov3.pth')
os.makedirs('out', exist_ok=True)
def extract_chunk(zpath):
    with zipfile.ZipFile(zpath) as zf:
        tops = {p.split('/')[0] for p in zf.namelist() if p.strip('/')}
        target = f"out/{os.path.splitext(os.path.basename(zpath))[0]}" if ('images' in tops or 'labels' in tops) else ('.' if tops=={'out'} else 'out')
        os.makedirs(target, exist_ok=True); zf.extractall(target)
for z in [p for p in sorted(glob.glob(f'{DRIVE_DIR}/*.zip')) if 'dataset' not in os.path.basename(p).lower()]:
    extract_chunk(z); print('extracted', os.path.basename(z))
print('out dirs:', len(glob.glob('out/*/labels')), '| train labels:', len(glob.glob('out/train/labels/*.json')))

In [ ]:
# ---- labeled CSVs from the JSON labels (img = out/<split>/images/<name>.jpg) ----
import json, glob, os, csv
def build_csv(split):
    rows=[]
    for j in glob.glob(f'out/{split}/labels/*.json'):
        d=json.load(open(j))
        if int(d.get('label',-1))<0: continue
        name=d.get('name', os.path.splitext(os.path.basename(j))[0]); img=f'out/{split}/images/{name}.jpg'
        if os.path.exists(img): rows.append({'path':img,'center':d.get('center',''),'class':'','label':int(d['label'])})
    with open(f'{split}_colab.csv','w',newline='') as f:
        w=csv.DictWriter(f,fieldnames=['path','center','class','label']); w.writeheader(); w.writerows(rows)
    print(f'{split}_colab.csv', len(rows),'pos=',sum(r['label'] for r in rows),'centers=',sorted({r['center'] for r in rows}))
build_csv('train')
try: build_csv('val')
except Exception as e: print('val skip', e)

In [ ]:
# ---- 35-concept target matrix (needed for Stage-1 concept pretraining — both backbones). Reuse from Drive if cached. ----
import os, shutil
os.makedirs('phase3/cache', exist_ok=True)
if os.path.exists('phase3/cache/concept_targets.npz'):
    print('concept_targets.npz present.')
elif os.path.exists(f'{DRIVE_DIR}/concept_targets.npz'):
    shutil.copy(f'{DRIVE_DIR}/concept_targets.npz', 'phase3/cache/concept_targets.npz'); print('REUSED concept_targets.npz')
else:
    !python -m phase3.build_concept_targets --out phase3/cache/concept_targets.npz
    shutil.copy('phase3/cache/concept_targets.npz', f'{DRIVE_DIR}/concept_targets.npz'); print('built concept_targets.npz')

## Stage-1 — concept-supervised pretraining (full-35 losses: certain / uncertain / smoothing)

LIGHT + L2-SP anchor → *shape*, don't overwrite SSL. Upgrades vs the old masked-BCE:
- **`--discrim full15`** — the full discriminative core (9 → 15; adds `color_heterogeneity, whitish_focal_area, vascular_irregularity, dilated_vessels, focal_abnormal_vessels, border_sharpness`). This is the real substance of "more labels".
- **certain** = class-balanced soft-BCE (`--pos_weight_cap 10` rebalances rare-positive concepts, e.g. `depression_ulceration` p=.055 → 10× — the term most likely to move PPV@90R).
- **uncertain** (`--unc_w 0.1`) = prior-pull on `not_assessable` cells instead of hard-masking (never invents a label).
- **smoothing** (`--smooth_eps 0.05`) = target shrinks toward the per-concept prior so the head can't get more confident than the noisy VLM warrants (anti center-memorization).

Every knob **defaults to the old masked-BCE** → clean superset, ablatable via the flags. Note: **7 of the 35 concepts are dead constants** (value ≡ 0: modality/distance/view/landmark/interpretable_fraction/dominant_color/lesion_size) and are auto-dropped; context/acquisition concepts go to a **detached AUX head** so "all 35" never re-injects center style into the trunk.

In [ ]:
# Stage-1: concept-supervised pretraining -> concept_encoder.pt (per-backbone cache; 30 epochs).
import os, shutil
CE_DRIVE = f'{DRIVE_DIR}/concept_encoder_{BACKBONE}.pt'
if os.path.exists(CE_DRIVE):
    shutil.copy(CE_DRIVE, 'concept_encoder.pt'); print(f'REUSED concept_encoder_{BACKBONE}.pt -> Stage-1 SKIPPED')
else:
    print(f'running Stage-1 ({BACKBONE}, 30 epochs) ...')
    !python -m phase3.pretrain_concept --targets phase3/cache/concept_targets.npz --backbone {BACKBONE} \
        --unfreeze 3 --epochs 30 --bs 128 --lr 1e-4 --grl 1.0 --l2sp 1.0 --workers 8 \
        --discrim full15 --context_route detach --certain_floor 0.7 --smooth_eps 0.05 --unc_w 0.1 --pos_weight_cap 10 \
        --out concept_encoder.pt
    shutil.copy('concept_encoder.pt', CE_DRIVE); print(f'saved concept_encoder_{BACKBONE}.pt')

## Stage-2 — FINAL ship: **GastroNet-DINOv2 @448** + concept-init + CG-AMIL attention-MIL + **color-aug OOD** + MixStyle + SEMI + WiSE-FT (3 seeds → ensemble)

In [ ]:
# ---- DE-RISK GATE (run ONCE before the single submission) — does the OOD bundle beat baseline on the NEW-center proxy? ----
# BASE   = dinov2@448 + concept-init + SEMI + mild aug + mean-pool  (the safe exps/2-style recipe)
# BUNDLE = BASE + --aug domain (color/FDA OOD) + --mixstyle + --cg-head (attention-MIL)   (the winning method)
# Trains both with each center held out; paired AUROC/AUPRC gate. Ships NOTHING. Set RUN_GATE=True to spend ~4 finetunes (~1h).
RUN_GATE = False
if RUN_GATE:
    import os, shutil, numpy as np, phase3.evaluate as ev
    os.makedirs('phase3/cache', exist_ok=True)
    if not os.path.exists('phase3/cache/unl_manifest.npz'):
        if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
            shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz')
        else:
            !python -m phase3.mine_hardneg --manifest-only
            shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz')
    SEMI = ('--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
            '--semi-n 300000 --semi-bs 192 --semi-steps 10')
    BASE   = f'--backbone dinov2 --img 448 --init concept_encoder.pt --unfreeze 6 --wise-ft 0.7 --epochs 12 --bs 96 --loss bce+rank+pauc --warmup 2 {SEMI}'
    BUNDLE = BASE + ' --aug domain --mixstyle --cg-head --attn-entropy 0.1 --attn-floor 0.5'
    for hold in ['center_2', 'center_1']:
        for name, flags in [('base', BASE), ('bundle', BUNDLE)]:
            print(f'--- holdout {hold} : {name} ---')
            !python -m phase3.finetune --train-csv train_colab.csv --seed 0 --holdout {hold} {flags} --out loco_{name}_{hold}.pt
    def leg(h, n):
        d = np.load(f'loco_{n}_{h}_loco.npz', allow_pickle=True); return d['y'], d['c'], d['s']
    fails = []
    for hold in ['center_2', 'center_1']:
        y, c, sb = leg(hold, 'bundle'); _, _, sa = leg(hold, 'base')
        print(f'\nholdout {hold}  (BUNDLE vs BASE):')
        for m in ('auroc', 'auprc'):
            g = ev.gate(y, sb, sa, center=c, metric=m, B=2000)
            if g['verdict'] == 'FAIL': fails.append((hold, m))
            print(f"  {m}: Δ={g['delta']:+.4f} CI[{g['lo']:+.4f},{g['hi']:+.4f}] -> {g['verdict']}")
    print('\nDECISION:', f'BUNDLE regresses at {fails} -> ship BASE (drop OOD levers)' if fails
          else 'BUNDLE holds/improves on BOTH new-center legs -> ship the winning bundle (cell 11 is correct)')
else:
    print('Gate SKIPPED. Recommend RUN_GATE=True once: it tells you if --aug domain/--mixstyle/--cg-head actually help the unseen center BEFORE you spend the submission.')


In [ ]:
# ==== FINAL SHIP: GastroNet-DINOv2@448 + VLM-concept-init + CG-AMIL + color-aug OOD + MixStyle + SEMI + WiSE-FT, 3 seeds -> ensemble ====
import os, shutil
# unl_manifest.npz = paths+suspicion for SEMI (built from the VLM pool; cache to Drive)
if not os.path.exists('phase3/cache/unl_manifest.npz'):
    os.makedirs('phase3/cache', exist_ok=True)
    if os.path.exists(f'{DRIVE_DIR}/unl_manifest.npz'):
        shutil.copy(f'{DRIVE_DIR}/unl_manifest.npz', 'phase3/cache/unl_manifest.npz'); print('REUSED unl_manifest.npz')
    else:
        !python -m phase3.mine_hardneg --manifest-only
        shutil.copy('phase3/cache/unl_manifest.npz', f'{DRIVE_DIR}/unl_manifest.npz'); print('built unl_manifest.npz')
# Phase-2: concept-init (both backbones), 15 epochs, attention-MIL head + SEMI.
# --attn-entropy 0.1 --attn-floor 0.5 = ONE-SIDED floor (penalize only if attention COLLAPSES; lets it FOCUS on lesion).
# --semi-bs 192 for 448 (~1.8x the 336 memory; raise if VRAM allows).
FLAGS = (f'--backbone {BACKBONE} --img 448 --init concept_encoder.pt --cg-head --attn-entropy 0.1 --attn-floor 0.5 '
         '--aug domain --mixstyle '                                   # color/FDA OOD aug (train+semi strong view) + param-free MixStyle center-invariance
         '--unfreeze 6 --wise-ft 0.7 --epochs 15 '
         '--semi-manifest phase3/cache/unl_manifest.npz --semi-weight 0.5 '
         '--semi-n 300000 --semi-bs 192 --semi-steps 10 --ema-decay 0.99')
for s in [0, 1, 2]:
    !python -m phase3.finetune --train-csv train_colab.csv --holdout none --seed {s} \
        {FLAGS} --bs 96 --loss bce+rank+pauc --warmup 2 --out ship_seed{s}.pt
for s in [0, 1, 2]: shutil.copy(f'ship_seed{s}.pt', f'{DRIVE_DIR}/ship_seed{s}.pt')
print(f'DONE -> ship_seed0/1/2.pt ({BACKBONE} + concept-init + CG-AMIL@448 + SEMI, 15ep) saved to Drive. Next: Package cell.')
# Safe fallbacks: drop --cg-head (mean-pool) ; drop --semi-manifest (no SEMI) ; --backbone dinov2 (exp1 0.018 path)

## Package the submission (offline container)

`ship_seed{0,1,2}.pt` are the final weights. To submit:
1. Copy them into the container: `RARE25-Submission/resources/ship_seed{0,1,2}.pt`.
2. The container (`model/viscera_model.py`) loads them and runs **5-view TTA (orig/hflip/vflip/rot90/rot270) + 3-seed prob-mean** offline (`--network none`, per-image) — **identical** to the val cell below.
3. Build → test → save: run `do_test_run.sh`, then `do_save.sh`. **Validate the SAVED tar** (`gunzip -c <tar> | docker load` and run that image) before uploading — that's the exact artifact the platform runs.

The val cell below is a **same-center smoke test** = a mirage (optimistic; it read 0.65 before the real 0.018). It only confirms inference works; the honest new-center number is the **leaderboard**.

## Val scoring → competition metric (PPV@90R @1% prevalence, bootstrap median + CI)
Scores `val_colab.csv` with the 3-seed ensemble and reports the leaderboard metric the same way the grader does (curve-point PPV@90R, 1% prevalence, bootstrap), plus AUROC/AUPRC. ⚠️ **This is SAME-CENTER (both centers were in training) → optimistic** (it read 0.65 vs the real new-center 0.018). Use it only as a smoke test that inference works; the honest new-center number comes from **RARE25-val / the leaderboard**, not here. Read the **CI**, not the point.

In [ ]:
# ---- score val with the 3-seed ensemble + 5-VIEW TTA (EXACTLY the shipped container), report the 5 metrics ----
import os, csv, shutil, numpy as np
# restore ensemble from Drive if the runtime was reset
for s in [0, 1, 2]:
    if not os.path.exists(f'ship_seed{s}.pt') and os.path.exists(f'{DRIVE_DIR}/ship_seed{s}.pt'):
        shutil.copy(f'{DRIVE_DIR}/ship_seed{s}.pt', f'ship_seed{s}.pt')
assert all(os.path.exists(f'ship_seed{s}.pt') for s in [0, 1, 2]), 'ship_seed*.pt missing — run the ship cell or copy from Drive'
assert os.path.exists('val_colab.csv'), 'val_colab.csv missing — run the CSV-builder cell (needs out/val)'

# --tta 5view = orig/hflip/vflip/rot90/rot270 + prob-mean = IDENTICAL to RARE25-Submission/model/viscera_model.py,
# so this val number is what the offline container actually produces (not the old hflip-only approximation).
!python -m phase3.infer --model ship_seed0.pt,ship_seed1.pt,ship_seed2.pt --csv val_colab.csv --tta 5view --out preds.csv

from phase3 import evaluate as ev
# preds.csv = name,score,label ; pull center from val_colab.csv by frame name
center_by = {os.path.splitext(os.path.basename(r['path']))[0]: r.get('center', '')
             for r in csv.DictReader(open('val_colab.csv'))}
P = list(csv.DictReader(open('preds.csv')))
y = np.array([int(r['label']) for r in P]); s = np.array([float(r['score']) for r in P])
cen = np.array([center_by.get(r['name'], '') for r in P])
print(f'val frames={len(y)}  pos={int(y.sum())}  centers={sorted(set(cen))}  (TTA=5view = the container)\n')

hdr = f"{'split':10s} {'n':>4s} {'pos':>3s} | {'PPV@90R':>8s} {'CI_low':>7s} {'CI_high':>7s} | {'AUROC':>6s} {'AUPRC':>6s}"
print(hdr); print('-' * len(hdr))


def line(tag, yv, sv, cv):
    r = ev.report_full(yv, sv, cv if len(set(cv)) > 1 else None, target=0.9, prevalence=0.01, B=2000)
    print(f"{tag:10s} {r['n']:>4d} {r['pos']:>3d} | {r['ppv90']:>8.3f} {r['ci_lo']:>7.3f} {r['ci_hi']:>7.3f} | "
          f"{r['auroc']:>6.3f} {r['auprc']:>6.3f}")


line('POOLED', y, s, cen)
for cc in sorted(set(cen)):
    mk = cen == cc
    if mk.sum() and y[mk].sum() > 0:
        line(cc, y[mk], s[mk], cen[mk])
print("\nTTA=5view here == the offline container, so val predicts the leaderboard preprocessing faithfully.")
print("PPV@90R = curve-point @1% prevalence (leaderboard metric, HIGH variance at few pos — read the CI).")
print("AUROC/AUPRC = threshold-free ranking, STABLE at few pos, but NOT the 1%-operating-point score.")
print("SAME-CENTER val is optimistic (ship saw both centers); the honest new-center number is RARE25-val / the leaderboard.")

## D2F+ — decorrelated CNN member (LP-FT: Stage-1 concept→convergence, Stage-2 FROZEN encoder)
A **ConvNeXt-T / ResNet50** member makes DIFFERENT cross-center mistakes than the ViT anchor (local texture vs patch-token style) → rank-averaging cancels per-family center bias (the winner's ResNet50⊕ViT lever). Design (Kumar LP-FT, matches our frozen-LP=0.929 evidence): Stage-1 concept-teaching to convergence → Stage-2 **head-only (frozen encoder)** → preserves center-agnostic concept features. **Gate on LOCO before trusting it.**

In [ ]:
# D2F-1: CNN Stage-1 — concept-supervised to CONVERGENCE on the 144k pool (ConvNeXt-T). Cache to Drive.
import os, shutil
ARCH = 'convnext_tiny'   # or 'resnet50' (winner's family)
CE = f'{DRIVE_DIR}/cnn_concept_{ARCH}.pt'
if os.path.exists(CE):
    shutil.copy(CE, 'cnn_concept.pt'); print('REUSED', CE)
else:
    !python -m phase3.cnn_member --stage concept --arch {ARCH} --targets phase3/cache/concept_targets.npz \
        --img 224 --epochs 30 --bs 128 --lr 1e-4 --out cnn_concept.pt   # 30ep = to convergence (watch loss plateau)
    shutil.copy('cnn_concept.pt', CE); print('saved', CE)

In [ ]:
# D2F-2: LOCO GATE — does the CNN member DECORRELATE + help on the held-out center? (head-only, both legs)
# Trains head-only CNN with each center held out, scores the held-out center, and checks Spearman vs the ViT anchor.
RUN_CNN_GATE = True
if RUN_CNN_GATE:
    import numpy as np, csv, torch
    from phase3.cnn_member import CNNMember
    from sklearn.metrics import roc_auc_score
    from scipy.stats import spearmanr
    for hold in ['center_2','center_1']:
        !python -m phase3.cnn_member --stage finetune --arch {ARCH} --init cnn_concept.pt --unfreeze-stages 0 \
            --train-csv train_colab.csv --holdout {hold} --img 224 --epochs 20 --bs 96 --out cnn_loco_{hold}.pt
    rows=[r for r in csv.DictReader(open('val_colab.csv'))]; y=np.array([int(r['label']) for r in rows])
    cen=np.array([r['center'] for r in rows]); from PIL import Image; imgs=[Image.open(r['path']) for r in rows]
    for hold in ['center_2','center_1']:
        m=cen==hold; s=CNNMember(f'cnn_loco_{hold}.pt').score_frames([imgs[i] for i in np.where(m)[0]])
        print(f'  CNN LOCO holdout {hold}: AUROC={roc_auc_score(y[m],s):.3f}  (compare to ViT-B ~0.85-0.93)')
    print('DECISION: keep the CNN member only if its LOCO AUROC is decent (>~0.80) AND it decorrelates (Spearman<0.9 vs ViT).')
else: print('CNN gate skipped.')

In [ ]:
# D2F-3: CNN SHIP (head-only, both centers) + D2F+ ENSEMBLE val score (ViT anchor ⊕ CNN, rank-average)
import numpy as np, csv, glob, torch
from PIL import Image
from phase3.cnn_member import CNNMember
if not os.path.exists('cnn_member.pt'):
    !python -m phase3.cnn_member --stage finetune --arch {ARCH} --init cnn_concept.pt --unfreeze-stages 0 \
        --train-csv train_colab.csv --holdout none --img 224 --epochs 20 --bs 96 --out cnn_member.pt
    shutil.copy('cnn_member.pt', f'{DRIVE_DIR}/cnn_member_{ARCH}.pt')
rows=[r for r in csv.DictReader(open('val_colab.csv'))]; y=np.array([int(r['label']) for r in rows])
cen=np.array([r['center'] for r in rows]); imgs=[Image.open(r['path']) for r in rows]
def rank01(x): o=np.argsort(x,kind='mergesort');r=np.empty_like(o,float);r[o]=np.arange(len(x));return r/max(len(x)-1,1)
def fpr90(yy,ss,R=.9):
    P=yy.sum();N=(yy==0).sum();o=np.argsort(-ss);ys=yy[o];tp=np.cumsum(ys);fp=np.cumsum(1-ys);rc=tp/P
    return fp[min(np.searchsorted(rc,R),len(rc)-1)]/N
ppv1=lambda f:.009/(.009+.99*f)
# ViT anchor scores (reuse the shipped 3-seed ensemble via infer.py's preds.csv, or recompute); here load preds.csv
vit = np.array([float(x) for x in open('preds.csv').read().splitlines()[1:]]) if os.path.exists('preds.csv') else None
cnn = CNNMember('cnn_member.pt').score_frames(imgs)
members={'CNN':cnn}; 
if vit is not None and len(vit)==len(y): members['ViT-anchor']=vit
# rank-average (protects the tail vs one miscalibrated member)
ens = np.mean([rank01(s) for s in members.values()],0)
print('members:', list(members.keys()))
for tag,mask in [('POOLED',np.ones(len(y),bool)),('center_2',cen=='center_2')]:
    for nm,s in {**members,'D2F+ ens':ens}.items():
        f=fpr90(y[mask],s[mask]); print(f'  {tag:8} {nm:12} PPV@90R={ppv1(f):.3f} AUROC={__import__("sklearn.metrics",fromlist=["roc_auc_score"]).roc_auc_score(y[mask],s[mask]):.3f}')
print('D2F+ wins only if the ensemble beats EACH member on center_2 (the honest leg). Gate on LOCO, not this same-center val.')